In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib seaborn prophet tensorflow joblib


In [ ]:
import warnings; warnings.filterwarnings("ignore")

import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler

from prophet import Prophet
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping
import joblib


In [ ]:
from google.colab import files
uploaded = files.upload()  # choose nasa_power_data_all_params.csv

import io
import pandas as pd
csv_name = [n for n in uploaded.keys() if n.endswith('.csv')][0]
df = pd.read_csv(io.BytesIO(uploaded[csv_name]))
df.head()

In [ ]:
                                                                                                                                                                                                                                                                                                     import matplotlib.pyplot as plt
import seaborn as sns
assert 'Date' in df.columns
assert 'ALLSKY_SFC_SW_DWN' in df.columns
print(df.shape, df.columns.tolist(), df.isnull().sum())
plt.figure(figsize=(10,7))
sns.heatmap(df.corr(numeric_only=True))
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
def metrics(y_true, y_pred):
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    return {
        "RMSE": mean_squared_error(y_true, y_pred, squared=False),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }

def plot_actual_pred(y_true, y_pred, title):
    plt.figure(figsize=(12,5))
    plt.plot(y_true.values, label="Actual")
    plt.plot(y_pred, label="Predicted")
    plt.title(title); plt.legend(); plt.show()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def metrics(y_true, y_pred):
    return {
        "RMSE": mean_squared_error(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }

def plot_actual_pred(y_true, y_pred, title):
    plt.figure(figsize=(12,5))
    plt.plot(y_true.values, label="Actual")
    plt.plot(y_pred, label="Predicted")
    plt.title(title); plt.legend(); plt.show()


rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
print("RF:", metrics(y_test, pred_rf))
plot_actual_pred(y_test, pred_rf, "Random Forest — Actual vs Predicted")

imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False).head(10)
imp.sort_values().plot(kind='barh'); plt.title("RF — Top 10 Features"); plt.show()

In [ ]:
from prophet import Prophet
prophet_df = df[['Date', target]].rename(columns={'Date':'ds', target:'y'})
split = int(0.8*len(prophet_df))
train_df, test_df = prophet_df.iloc[:split], prophet_df.iloc[split:]

m = Prophet(daily_seasonality=True, yearly_seasonality=True, weekly_seasonality=False)
m.fit(train_df)
future = m.make_future_dataframe(periods=len(test_df), freq='D')
fc = m.predict(future)

y_pred_prophet = fc.iloc[-len(test_df):]['yhat'].values
print("Prophet:", metrics(test_df['y'].values, y_pred_prophet))

m.plot(fc); plt.title("Prophet Forecast"); plt.show()
m.plot_components(fc); plt.show()

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


def metrics(y_true, y_pred):
    return {
        "RMSE": mean_squared_error(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }

def plot_actual_pred(y_true, y_pred, title):
    plt.figure(figsize=(12,5))
    plt.plot(y_true, label="Actual")
    plt.plot(y_pred, label="Predicted")
    plt.title(title); plt.legend(); plt.show()


series = df[[target]].values
scaler = MinMaxScaler((0,1))
series_scaled = scaler.fit_transform(series)

def make_seq(data, window=7):
    Xs, ys = [], []
    for i in range(len(data)-window):
        Xs.append(data[i:i+window]); ys.append(data[i+window])
    return np.array(Xs), np.array(ys)

WINDOW = 7
X_seq, y_seq = make_seq(series_scaled, WINDOW)
split_seq = int(0.8*len(X_seq))
Xtr, Xte = X_seq[:split_seq], X_seq[split_seq:]
ytr, yte = y_seq[:split_seq], y_seq[split_seq:]

model = Sequential([LSTM(64, activation='tanh', input_shape=(WINDOW,1)), Dense(1)])
model.compile(optimizer='adam', loss='mse')
es = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model.fit(Xtr, ytr, validation_split=0.2, epochs=25, batch_size=16, callbacks=[es], verbose=1)

pred_lstm = model.predict(Xte)
pred_lstm = scaler.inverse_transform(pred_lstm).flatten()
yte_inv = scaler.inverse_transform(yte).flatten()
print("LSTM:", metrics(yte_inv, pred_lstm))
plot_actual_pred(pd.Series(yte_inv), pred_lstm, "LSTM — Actual vs Predicted")

In [ ]:
import joblib
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
import tensorflow as tf

# Load the saved Random Forest model and features
rf_model = joblib.load('rf_model.joblib')
with open('rf_features.txt', 'r') as f:
    FEATURES = f.read().split("\\n")

# Make predictions using the loaded Random Forest model
# Ensure X_test is defined and contains the FEATURES used for training
# Assuming X_test is available from previous cells
y_pred_rf = rf_model.predict(X_test[FEATURES])


# Load the saved Prophet model
prophet_model = joblib.load('prophet_model.joblib')

# Re-calculate y_pred_prophet
# Assuming test_df is available from previous cells and has 'ds' column
future = prophet_model.make_future_dataframe(periods=len(test_df), freq='D')
fc = prophet_model.predict(future)
y_pred_prophet = fc.iloc[-len(test_df):]['yhat'].values


# Load the saved LSTM model and scaler
lstm_model = tf.keras.models.load_model('lstm_model.keras')
lstm_scaler = joblib.load('lstm_scaler.joblib')

# Re-calculate yte_inv (y_test_rescaled)
# Assuming yte is available from the LSTM cell
# Need to ensure yte is the scaled test target values used for LSTM
# If yte is not available, we'll need to re-create it.
# For now, let's assume yte is available and corresponds to the test set used for LSTM
# Re-scaling the original y_test using the LSTM scaler
y_test_rescaled = lstm_scaler.inverse_transform(yte).flatten()
pred_lstm_scaled = lstm_model.predict(Xte)
y_pred_lstm = lstm_scaler.inverse_transform(pred_lstm_scaled).flatten()


# Collect results using the re-calculated predictions and the original y_test
results = {
    "Random Forest": {
        "RMSE": mean_squared_error(y_test, y_pred_rf),
        "R2": r2_score(y_test, y_pred_rf)
    },
    "Prophet": {
        "RMSE": mean_squared_error(test_df['y'].values, y_pred_prophet),
        "R2": r2_score(test_df['y'].values, y_pred_prophet)
    },
    "LSTM": {
        "RMSE": mean_squared_error(y_test_rescaled, y_pred_lstm),
        "R2": r2_score(y_test_rescaled, y_pred_lstm)
    }
}

results_df = pd.DataFrame(results).T
print(results_df)